# Feast offline-to-online training and inference

This notebook runs a production-shaped, CPU-only feature pipeline:

1. a Spark Operator `SparkApplication` generates a synthetic batch dataset and stores it as partitioned Parquet in S3-compatible object storage;
2. Spark performs Feast's point-in-time historical join and trains a small NumPy model;
3. Feast uses PostgreSQL for its durable SQL registry and materializes the latest feature values into Redis; and
4. KServe deploys a model server that reads Redis-backed online features before predicting.

Object storage is the durable, versionable offline data layer; operator-managed Spark provides elastic batch compute; PostgreSQL stores Feast metadata rather than feature history; and Redis serves low-latency online lookups. Feast classifies its Spark offline store as a contributed integration without full test coverage, so qualify it against your scale and upgrade requirements or use a fully supported warehouse while retaining the same S3-and-Spark data pipeline.

The reusable source code and Kubernetes manifests are in `assets/feast-offline-to-online-inference/`. Downloading only this `.ipynb` will not include those required files. From a Workbench terminal, use a sparse checkout to obtain the notebook and asset bundle together:

```bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/alauda/aml-docs.git
git -C aml-docs sparse-checkout set --no-cone \
  /docs/en/train/guides/feast-offline-to-online-inference.ipynb \
  /docs/en/train/guides/assets/feast-offline-to-online-inference/
cd aml-docs/docs/en/train/guides
```

Start the notebook from that `guides` directory so the default asset path resolves. Set `FEAST_ASSET_DIR` only when your Workbench uses a different working directory.

Prerequisites:

- the Alauda Spark Operator is installed through OLM and `sparkapplications.sparkoperator.k8s.io` exists;
- `FEAST_SPARK_IMAGE` is set to a Spark runtime containing PySpark, Feast 0.61.x with Spark and Redis support, NumPy, pandas, PyArrow, PyYAML, and a Hadoop S3A connector compatible with the image's Hadoop version;
- a `feast-data-stores` Secret in `feast-demo` with `redis` and `sql` keys;
- a `feast-s3-credentials` Secret in `feast-demo` with `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_ENDPOINT_URL`, and `S3_BUCKET`;
- a pre-created S3 bucket, a default ReadWriteOnce storage class, and the Feast and KServe operators; and
- `bash`, `kubectl`, `sed`, and `curl` in the Workbench image.

Find the global-cluster registry without copying an internal registry hostname from this document:

```bash
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
```

Choose an approved Spark runtime from that registry and set its complete reference in `FEAST_SPARK_IMAGE`. Optionally set `FEAST_SPARK_VERSION` and `FEAST_MODEL_IMAGE`; the commands default to Spark 4.0.1 metadata and the Feast 0.61.0 model-server repository in the discovered registry.


## 1. Verify the operator and asset bundle

The Spark runtime image contains Spark and its S3A client libraries. No separate Spark or Hadoop service is installed by this example.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
test -f "$ASSET_DIR/batch.py"
test -f "$ASSET_DIR/spark-application.yaml"
: "${FEAST_SPARK_IMAGE:?Set FEAST_SPARK_IMAGE to an approved global-registry image}"
kubectl get crd sparkapplications.sparkoperator.k8s.io
kubectl get crd featurestores.feast.dev
kubectl get crd inferenceservices.serving.kserve.io
kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}'
echo


## 2. Prepare PostgreSQL registry and Redis online serving

The Feast Operator manages the control plane and online-serving backends. PostgreSQL stores the SQL registry and Redis stores materialized online feature values. Spark reads the offline Parquet history directly from S3.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
kubectl apply -f "$ASSET_DIR/namespaces.yaml"
kubectl -n feast-demo get secret feast-data-stores
kubectl -n feast-demo get secret feast-s3-credentials
kubectl apply -f "$ASSET_DIR/feature-store.yaml"

for _ in $(seq 1 60); do
  phase="$(kubectl -n feast-demo get featurestore feast-notebook \
    -o jsonpath='{.status.phase}' 2>/dev/null || true)"
  echo "${phase:-Pending}"
  [ "$phase" = Ready ] && break
  if [ "$phase" = Failed ]; then
    kubectl -n feast-demo describe featurestore feast-notebook
    exit 1
  fi
  sleep 10
done
[ "${phase:-}" = Ready ] || { echo "FeatureStore did not become Ready" >&2; exit 1; }
kubectl -n feast-demo get service feast-notebook-online -o jsonpath='{.spec.ports[*].name}{"\n"}' | grep -qw metrics
kubectl -n feast-demo get featurestore feast-notebook


## 3. Submit the synthetic S3 batch as a SparkApplication

The code ConfigMap packages `batch.py` and `server.py` from the asset directory. The Spark driver configures S3A through the public `SparkSession.builder.config()` API before creating the Spark context. S3A reads credentials from the Secret-backed driver and executor environments, so the keys are not written into the `SparkApplication` or model PVC.

The application generates deterministic synthetic driver events, writes partitioned Parquet to S3, registers a Feast `SparkSource`, performs the historical join, trains the sample model, and materializes the same features into Redis.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
: "${FEAST_SPARK_IMAGE:?Set FEAST_SPARK_IMAGE to an approved global-registry image}"
SPARK_VERSION="${FEAST_SPARK_VERSION:-4.0.1}"

kubectl -n feast-demo create configmap feast-offline-batch-code \
  --from-file=batch.py="$ASSET_DIR/batch.py" \
  --from-file=server.py="$ASSET_DIR/server.py" \
  --dry-run=client -o yaml | kubectl apply -f -
kubectl apply -f "$ASSET_DIR/spark-rbac.yaml"
kubectl apply -f "$ASSET_DIR/model-pvc.yaml"
kubectl -n feast-demo delete sparkapplication feast-offline-batch \
  --ignore-not-found --wait=true
sed \
  -e "s|FEAST_SPARK_IMAGE_PLACEHOLDER|$FEAST_SPARK_IMAGE|g" \
  -e "s|FEAST_SPARK_VERSION_PLACEHOLDER|$SPARK_VERSION|g" \
  "$ASSET_DIR/spark-application.yaml" | kubectl apply -f -

state=""
for _ in $(seq 1 120); do
  state="$(kubectl -n feast-demo get sparkapplication feast-offline-batch \
    -o jsonpath='{.status.applicationState.state}' 2>/dev/null || true)"
  echo "${state:-SUBMITTED}"
  case "$state" in
    COMPLETED|FAILED|FAILED_SUBMISSION|INVALIDATING|UNKNOWN) break ;;
  esac
  sleep 10
done
driver="$(kubectl -n feast-demo get sparkapplication feast-offline-batch \
  -o jsonpath='{.status.driverInfo.podName}' 2>/dev/null || true)"
if [ "$state" != COMPLETED ]; then
  [ -n "$driver" ] && kubectl -n feast-demo logs "$driver" --tail=300 || true
  exit 1
fi
[ -n "$driver" ] && kubectl -n feast-demo logs "$driver" --tail=100


## 4. Inspect the offline-to-online result

The completed Spark driver writes a small online-feature verification sample and the model artifact to the PVC. This short-lived inspector reads the sample without embedding it in the notebook.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
MODEL_IMAGE="${FEAST_MODEL_IMAGE:-}"
if [ -z "$MODEL_IMAGE" ]; then
  registry="$(kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}')"
  MODEL_IMAGE="$registry/mlops/feast/feature-server:0.61.0"
fi
kubectl -n feast-demo delete pod feast-model-inspector --ignore-not-found --wait=true
sed "s|FEAST_MODEL_IMAGE_PLACEHOLDER|$MODEL_IMAGE|g" \
  "$ASSET_DIR/model-inspector.yaml" | kubectl apply -f -
kubectl -n feast-demo wait \
  "--for=jsonpath={.status.phase}=Succeeded" pod/feast-model-inspector --timeout=180s
kubectl -n feast-demo logs feast-model-inspector
kubectl -n feast-demo delete pod feast-model-inspector --wait=true


## 5. Deploy the Redis-backed model with KServe

The model server loads the weights and serving-only Feast configuration from the PVC, queries Redis through the operator-managed Feast online service, and exposes KServe's v2 inference protocol.


In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_ASSET_DIR:-assets/feast-offline-to-online-inference}"
MODEL_IMAGE="${FEAST_MODEL_IMAGE:-}"
if [ -z "$MODEL_IMAGE" ]; then
  registry="$(kubectl get configmap global-info -n kube-public -o jsonpath='{.data.registryAddress}')"
  MODEL_IMAGE="$registry/mlops/feast/feature-server:0.61.0"
fi
sed "s|FEAST_MODEL_IMAGE_PLACEHOLDER|$MODEL_IMAGE|g" \
  "$ASSET_DIR/serving-runtime.yaml" | kubectl apply -f -
kubectl apply -f "$ASSET_DIR/inference-service.yaml"
kubectl -n feast-demo get inferenceservice feast-online-model


## 6. Send an online-feature prediction

After the predictor has an available replica, send driver IDs to the KServe v2 endpoint. The server retrieves their materialized features from Redis and combines those values with the model trained by the `SparkApplication`.


In [ ]:
%%bash
set -euo pipefail
available=""
for _ in $(seq 1 90); do
  available="$(kubectl -n feast-demo get deployment feast-online-model-predictor \
    -o jsonpath='{.status.availableReplicas}' 2>/dev/null || true)"
  echo "availableReplicas=${available:-0}"
  [ "${available:-0}" -ge 1 ] 2>/dev/null && break
  sleep 10
done
[ "${available:-0}" -ge 1 ] 2>/dev/null || exit 1

kubectl -n feast-demo port-forward service/feast-online-model-predictor 18080:80 \
  >/tmp/feast-model-port-forward.log 2>&1 &
port_forward_pid=$!
trap 'kill "$port_forward_pid" 2>/dev/null || true' EXIT
for _ in $(seq 1 30); do
  curl -fsS http://127.0.0.1:18080/v2/health/ready >/dev/null 2>&1 && break
  sleep 2
done
curl -fsS -X POST \
  http://127.0.0.1:18080/v2/models/feast-online-model/infer \
  -H 'Content-Type: application/json' \
  -d '{"inputs":[{"name":"driver_id","shape":[2],"datatype":"INT64","data":[1,2]}]}'
echo


## 7. Benchmark the online feature service

This smoke benchmark measures the HTTPS Feast online-feature API through the Kubernetes Service, rather than timing the `/health` endpoint. It sends 200 requests at concurrency 20 and checks both HTTP status and that every returned feature status is `PRESENT`. The generated service certificate is cluster-local, so the test client disables certificate verification; use trusted CA verification for production clients.

The request contains three features (`conv_rate`, `acc_rate`, and `avg_daily_trips`) for four driver IDs. Set `FEAST_PERF_REQUESTS` and `FEAST_PERF_CONCURRENCY` to change the test size.

### Recorded x86 result

| Online backend | Connection mode | Throughput | Average | P50 | P95 | HTTP errors | Complete responses |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| SQLite file (baseline) | New HTTPS connection/request | 47.24 req/s | 405.5 ms | 242.9 ms | 1,652.2 ms | 2/200 | 196/200 |
| Redis 7.2 | New HTTPS connection/request | 138.49 req/s | 139.5 ms | 132.0 ms | 213.8 ms | 0/200 | 200/200 |
| Redis 7.2 | One persistent connection/worker | 218.69 req/s | 83.2 ms | 77.1 ms | 150.9 ms | 0/200 | 200/200 |

The SQLite result was not stable under concurrency: a separate run reached 86.54 req/s but returned one incomplete response, and later runs logged `sqlite3.InterfaceError: bad parameter or other API misuse`. Redis removed those read errors in this test. These numbers are a development-cluster comparison, not an SLA or capacity limit. Repeat the test with production-sized data, replicas, client connection pooling, and the target traffic shape before setting an SLA.


In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${FEAST_NAMESPACE:-feast-demo}"
FEATURESTORE="${FEAST_FEATURESTORE:-demo}"
REQUESTS="${FEAST_PERF_REQUESTS:-200}"
CONCURRENCY="${FEAST_PERF_CONCURRENCY:-20}"
SERVICE_HOST="feast-$FEATURESTORE-online.$NAMESPACE.svc.cluster.local"
kubectl exec -i -n "$NAMESPACE" "deploy/feast-$FEATURESTORE" -c online -- \
  env PERF_HOST="$SERVICE_HOST" PERF_REQUESTS="$REQUESTS" PERF_CONCURRENCY="$CONCURRENCY" python - <<'PY'
import concurrent.futures
import http.client
import json
import math
import os
import ssl
import statistics
import threading
import time

host = os.environ['PERF_HOST']
requests = int(os.environ.get('PERF_REQUESTS', '200'))
concurrency = int(os.environ.get('PERF_CONCURRENCY', '20'))
body = json.dumps({
    'features': [
        'driver_hourly_stats:conv_rate',
        'driver_hourly_stats:acc_rate',
        'driver_hourly_stats:avg_daily_trips',
    ],
    'entities': {'driver_id': [1001, 1002, 1003, 1005]},
}).encode()
headers = {'Content-Type': 'application/json', 'Content-Length': str(len(body))}
context = ssl._create_unverified_context()

def request():
    started = time.perf_counter()
    connection = http.client.HTTPSConnection(host, 443, context=context, timeout=10)
    try:
        connection.request('POST', '/get-online-features', body, headers)
        response = connection.getresponse()
        payload = json.loads(response.read())
        statuses = [
            status
            for result in payload.get('results', [])
            if isinstance(result, dict)
            for status in result.get('statuses', [])
        ]
        complete = len(statuses) == 16 and all(status == 'PRESENT' for status in statuses)
        return (time.perf_counter() - started) * 1000, response.status, complete, None
    except Exception as error:
        return (time.perf_counter() - started) * 1000, None, False, repr(error)
    finally:
        connection.close()

for _ in range(min(10, requests)):
    request()
barrier = threading.Barrier(concurrency)
per_worker = [requests // concurrency + (index < requests % concurrency) for index in range(concurrency)]

def worker(index):
    barrier.wait()
    return [request() for _ in range(per_worker[index])]

started = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
    results = [item for batch in pool.map(worker, range(concurrency)) for item in batch]
results = results[:requests]
elapsed = time.perf_counter() - started
latencies = sorted(result[0] for result in results)
errors = [result for result in results if result[1] != 200]
def percentile(value):
    index = min(len(latencies) - 1, max(0, math.ceil(value * len(latencies)) - 1))
    return latencies[index]
print(json.dumps({
    'target': f'https://{host}:443/get-online-features',
    'requests': len(results),
    'concurrency': concurrency,
    'throughput_requests_per_second': round(len(results) / elapsed, 2),
    'latency_ms': {
        'avg': round(statistics.mean(latencies), 3),
        'p50': round(percentile(0.50), 3),
        'p95': round(percentile(0.95), 3),
        'p99': round(percentile(0.99), 3),
    },
    'http_200': len(results) - len(errors),
    'errors': len(errors),
    'complete_feature_responses': sum(result[2] for result in results),
    'error_samples': [result[3] for result in errors[:3]],
}, indent=2))
PY


## 8. Monitor Feast online-service metrics

The online server exposes Prometheus metrics when `services.onlineStore.server.metrics: true` is set. The operator adds an HTTP Service port named `metrics` on port `8000`; this is separate from the TLS feature API on port `443`. Feast 0.61 exposes request count (`feast_online_features_request_total`), request entity-count histogram (`feast_online_features_entity_count`), and process CPU/memory gauges. This endpoint is operational telemetry, not a full drift detector; feature freshness and value distributions require a separate materialization/quality pipeline.

For a local check, port-forward the metrics port and inspect the exposition format:

```bash
kubectl -n feast-demo port-forward service/feast-notebook-online 18000:8000
curl -fsS http://127.0.0.1:18000/metrics | head
```

In a Prometheus Operator installation, scrape the Service port named `metrics` with a `ServiceMonitor` (or configure the equivalent static scrape). Example PromQL for a dashboard and alert is:

```promql
rate(feast_online_features_request_total[5m])
histogram_quantile(0.95, sum by (le) (rate(feast_online_features_entity_count_bucket[5m])))
feast_feature_server_memory_usage > 90
```

Metric names can vary with the Feast image version; discover the exact names with `/metrics` before creating recording rules. Alert on scrape failures, a sustained increase in entity-count or request rate beyond capacity, and process resource saturation. Track materialization freshness separately because it is not emitted by this Feast 0.61 endpoint.


In [ ]:
%%bash
set -euo pipefail
kubectl -n feast-demo get service feast-notebook-online -o jsonpath='{.spec.ports[?(@.name=="metrics")].port}{"\n"}'
kubectl -n feast-demo port-forward service/feast-notebook-online 18000:8000 >/tmp/feast-metrics-port-forward.log 2>&1 &
port_forward_pid=$!
trap 'kill "$port_forward_pid" 2>/dev/null || true' EXIT
for _ in $(seq 1 20); do curl -fsS http://127.0.0.1:18000/metrics >/tmp/feast-metrics.txt && break; sleep 1; done
test -s /tmp/feast-metrics.txt
rg '^(# HELP|# TYPE|feast_)' /tmp/feast-metrics.txt | head -40


## 9. Production Redis for an online-feature SLA

The benchmark above used a deliberately small, single-replica Redis instance with memory-only storage. Do not use that pattern for a production Feast online service: pod loss removes all materialized values, and it provides no failover, backup, TLS, monitoring, or capacity guarantee.

For production, deploy the platform-managed [Alauda Cache Service for Redis OSS (ACP documentation)](appservice/redis/redis.html) through ACP Data Services. Use that document to choose the Redis architecture and configure durable storage, authentication/TLS, resource sizing, high availability, backup/restore, monitoring, and the read-write access endpoint. Validate connectivity from the Feast namespace before materialization.

For the Feast-side Secret layout and Redis online-store configuration, see [Redis online store + SQL registry](../../../develop/components/feast/quickstart.mdx#redis-online-store--sql-registry). Keep the registry durable as well (for example, SQL-backed PostgreSQL), materialize into the managed Redis service, and size replicas and connection pools against the target p95/p99 latency and availability objectives.
